# Рекомендации фильмов: ItemKNN, EASE и SASRec

**HSE RecSys Challenge 2026.** По истории оценок выбираем 10 фильмов для каждого пользователя.
Основная метрика — Recall@10. Подтверждённый Kaggle score EASE: **0,0860**, выше baseline_5 = 0,0821.

Тетрадка — воспроизводимый интерфейс к реализации в `src/`. Формулы, условия оценки и инструкции
находятся в `README.md`; история гипотез и сабмитов — в `EXPERIMENTS.md`.
Для запуска требуется весь проект, включая `src/`, `config.json` и четыре CSV.

## 1. Режим запуска

`auto` загружает SASRec, если checkpoint добавлен; иначе воспроизводит классические модели на CPU.
`train` выполняет полный эксперимент на GPU. `predict` создаёт CSV из готовой модели.
Установите зависимости из `requirements.txt` перед первым запуском.

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'src' / 'project.py').is_file():
    candidates = [p.parent for p in Path('/kaggle/input').rglob('config.json')
                  if (p.parent / 'src' / 'project.py').is_file()] if Path('/kaggle/input').exists() else []
    if len(candidates) != 1:
        raise FileNotFoundError('Откройте тетрадку из корня проекта или подключите весь проект как Kaggle Dataset')
    PROJECT_DIR = candidates[0]
sys.path.insert(0, str(PROJECT_DIR))

from src.project import load_config, run
from src.data import load_data
import pandas as pd
from IPython.display import display

MODE = 'auto'          # auto / baselines / train / predict
MODEL = 'sasrec'       # sasrec / ease; используется в predict
DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_DIR = Path('/kaggle/working/outputs') if Path('/kaggle/working').exists() else PROJECT_DIR / 'outputs'
CHECKPOINT = None      # Можно указать скачанный sasrec_full.pt
if CHECKPOINT is None and (PROJECT_DIR / 'outputs/checkpoints/sasrec.pt').is_file():
    CHECKPOINT = PROJECT_DIR / 'outputs/checkpoints/sasrec.pt'
config = load_config(PROJECT_DIR / 'config.json')
print('Режим:', MODE, '| данные:', DATA_DIR.name, '| результаты:', OUTPUT_DIR.name)

Режим: auto | данные: data | результаты: outputs


## 2. Данные и протокол оценки

Последние 20% событий каждого пользователя выделяются в validation. Все отложенные оценки — target;
бинарная train-матрица отражает факт взаимодействия. Случайные негативы для оценки не используются.
Из top-10 исключается вся известная история. Это локальная схема, а не подтверждённая копия скрытого теста Kaggle.

Признаки пользователей и жанров в выбранные модели не включаются. `timestamp` используется для порядка внутри пользователя.

In [2]:
data = load_data(DATA_DIR)
display(data.summary())
display(data.events.head())

,value
events,894149.0
users,6040.0
catalog_items,3706.0
items_with_events,3690.0
minimum_history,14.0
median_history,84.5


,user_id,item_id,rating,timestamp
0,0,1505,4,0
1,0,3669,3,1
2,0,584,4,2
3,0,3390,3,3
4,0,2885,4,4


## 3. Эксперимент

Популярность задаёт точку отсчёта; ItemKNN использует похожие фильмы; EASE обучает матрицу связей
с нулевой диагональю. Для EASE проверяются λ = 100, 300, 1000, 3000, 10000.
SASRec использует causal transformer RePlay и полную cross-entropy по каталогу.

Параметры всех моделей находятся в `config.json`. В полном `train` лучшая эпоха SASRec выбирается
по validation, затем новая модель обучается на всей истории ровно это число эпох.
`predict` не повторяет обучение SASRec. EASE в режиме predict быстро рассчитывается с фиксированной λ = 1000.

In [3]:
report = run(MODE, MODEL, DATA_DIR, OUTPUT_DIR, config, checkpoint=CHECKPOINT)
print('Завершён режим:', report['mode'])
print('Использованная модель:', report.get('selected_model'))
print('Файл результата:', report.get('submission'))

Auto mode: baselines. SASRec checkpoint: not added
popularity: Recall@10=0.036167
itemknn: Recall@10=0.055767
ease_l2_100: Recall@10=0.061258
ease_l2_300: Recall@10=0.063336
ease_l2_1000: Recall@10=0.065038
ease_l2_3000: Recall@10=0.064884
ease_l2_10000: Recall@10=0.062299
Завершён режим: baselines
Использованная модель: ease_l2_1000
Файл результата: submission_ease.csv


## 4. Результаты

Ниже показывается таблица текущего запуска, если он включал валидацию.
Для запуска только инференса используется отдельно обозначенная таблица ранее проведённых экспериментов.
Kaggle score берётся из журнала реальных сабмитов, а не из локальной метрики.

In [4]:
metrics_path = OUTPUT_DIR / 'metrics.csv'
if not metrics_path.exists():
    metrics_path = PROJECT_DIR / 'outputs/reference_metrics.csv'
    print('Справочные результаты прежнего запуска; не оценка текущего inference.')
display(pd.read_csv(metrics_path).sort_values('local_recall_at_10', ascending=False))

history_path = OUTPUT_DIR / 'sasrec_validation_history.csv'
if not history_path.exists():
    history_path = PROJECT_DIR / 'outputs/sasrec_validation_history.csv'
if history_path.exists():
    import matplotlib.pyplot as plt
    history = pd.read_csv(history_path)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    history.plot(x='epoch', y='train_loss', ax=axes[0], legend=False, title='SASRec: train loss')
    history.plot(x='epoch', y='local_recall_at_10', ax=axes[1], legend=False, title='SASRec: Recall@10')
    plt.tight_layout()
    plt.show()

,model,parameters_json,local_recall_at_10,evaluated_users,elapsed_seconds
0,ease_l2_1000,"{""regularization"": 1000.0}",0.065038,6040,5.067
1,ease_l2_3000,"{""regularization"": 3000.0}",0.064884,6040,4.773
2,ease_l2_300,"{""regularization"": 300.0}",0.063336,6040,5.094
3,ease_l2_10000,"{""regularization"": 10000.0}",0.062299,6040,4.566
4,ease_l2_100,"{""regularization"": 100.0}",0.061258,6040,4.863
5,itemknn,"{""neighbors"": 100, ""shrinkage"": 10.0}",0.055767,6040,3.302
6,popularity,{},0.036167,6040,1.435


## 5. Сабмит и сохранённая модель

Код проверяет состав пользователей, количество и уникальность рекомендаций, допустимые ID,
отсутствие просмотренного и повторное чтение CSV. Порядок пользователей совпадает с sample.
`submission.csv` содержит выбранный результат, а `submission_sasrec.csv` и `submission_ease.csv` сохраняют отдельные модели.

Два checkpoint имеют разное назначение: `checkpoints/sasrec_validation_best.pt` — для локальной оценки,
`checkpoints/sasrec.pt` — финальная модель для рекомендаций. Сохраняются веса и порядок ID, а не объект Python-модели.

In [5]:
submission_path = OUTPUT_DIR / report['submission']
display(pd.read_csv(submission_path, dtype=str).head())
print('Сабмит в outputs:', submission_path.name)
print('Все файлы запуска:', sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_file()))

Сабмит в outputs: submission_ease.csv
Все файлы запуска: ['metrics.csv', 'reference_metrics.csv', 'reference_run.json', 'run_report.json', 'submission.csv', 'submission_ease.csv']


,user_id,item_id
0,0,1811 2256 2732 331 2397 1001 3059 1808 3298 1248
1,1,3529 3656 1039 1686 1868 232 1491 1699 208 106
2,2,1956 1831 1560 2342 2338 463 2354 293 2256 452
3,3,1617 2194 3327 3318 3022 2186 2342 3435 3358 3005
4,4,1560 3677 2402 1337 3409 1809 1315 2862 2222 2774


## 6. Выводы

- Персонализация ItemKNN улучшила локальный Recall@10 относительно популярности: 0,03617 → 0,05577.
- EASE с λ = 1000 улучшил локальную метрику до 0,06504 и получил 0,0860 на Kaggle, превысив baseline_5.
- В GPU-прогоне SASRec локальный Recall@10 к концу обучения был около 0,1415.
  Точный максимум и выбранная эпоха фиксируются историей запуска; public score требует отдельного сабмита.
- Нельзя напрямую сравнивать локальную и Kaggle-метрики. Выбор модели производится на validation,
  а результат скрытого теста фиксируется отдельно в журнале.

Запуск из терминала: `python run.py auto`. Полное обучение: `python run.py train`.
Повторный инференс по весам: `python run.py predict --model sasrec`.